# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the `FAIR^2` dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. All references to dataset elements use their Croissant `@id` for precise identification and reproducibility.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install mlcroissant if needed (uncomment in Colab/Jupyter)
!pip install mlcroissant>=1.1 pandas

## 1. Data Loading

Load the dataset metadata and explore its basic descriptors. All data access is based on the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")


## 2. Data Overview

Croissant datasets may have multiple *record sets* (`cr:RecordSet`), each with their own unique `@id`.

Below, we list all available record sets in this dataset by their `@id` and display their field `@id`s and data columns. This information guides downstream extraction and analysis steps.

In [ ]:
# List all record sets and their fields using @id
record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record_set(s):")

for rec in record_sets:
    print(f"\nRecordSet @id: {rec.id}")
    print(f"  Name: {rec.name}")
    print(f"  Fields:")
    for field in rec.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, type: {field.data_type}")
    print(f"  File objects/columns:")
    for fo in rec.file_objects:
        print(f"    - FileObject @id: {fo.id}, columns: {[col.id for col in fo.columns]}")

## 3. Data Extraction

We use the record set `@id` to extract its data with `mlcroissant.Dataset.records(record_set=<@id>)`. Use the output above to select record set(s) to load. In this dataset, due to its focus on protein abundance, we expect a primary record set for proteins.

Below, each loaded DataFrame is indexed by the record set `@id`.

In [ ]:
# Collect all record set @id's for extraction
record_set_ids = [rec.id for rec in record_sets]
dataframes = {}

for rec_id in record_set_ids:
    print(f"Loading records for RecordSet {rec_id} ...")
    rec_records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(rec_records)
    dataframes[rec_id] = df
    print(f"  --> {df.shape[0]} records, columns: {list(df.columns)}\n")

if record_set_ids:
    primary_recset = record_set_ids[0]  # Example: use the first record set
    print(f"Fields (by @id) in record set '{primary_recset}':\n{list(dataframes[primary_recset].columns)}")
    display(dataframes[primary_recset].head())

## 4. Exploratory Data Analysis (EDA)

We now select a numeric field (accessed by its `@id`) for demonstration. We filter and normalize the field, referencing other columns by their Croissant `@id`.

*Adapt the field `@id`s and thresholds below according to the record set structures displayed above.*

In [ ]:
# Choose the record set and field by their @id
record_set_id = record_set_ids[0]  # Use the first found; edit as needed
df = dataframes[record_set_id]

# Find a numeric (float/integer) field by @id
numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
if not numeric_candidates:
    # Try to convert likely numeric columns
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            continue
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

print(f"Numeric fields (by @id): {numeric_candidates}")
numeric_field = numeric_candidates[0]  # Choose the first for demonstration

# Filtering (e.g., keep rows with value > threshold)
threshold = df[numeric_field].mean()  # use mean as an example threshold
filtered_df = df[df[numeric_field] > threshold].copy()

print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
display(filtered_df[[numeric_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} (first 5 rows):")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by another field if a categorical field exists
cat_fields = [c for c in df.columns if pd.api.types.is_string_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c])]
if cat_fields:
    group_field = cat_fields[0]
    grouped = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped means of {numeric_field} by {group_field} (by @id):")
    display(grouped.head())

## 5. Visualization

Visualize the distribution of a numeric column (`@id`) and relationships between two fields, using `matplotlib` and `seaborn` for demonstration. All field references are by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot of the main numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.xlabel(numeric_field + ' (@id)')
plt.title(f"Distribution of {numeric_field}")
plt.show()

# If a second numeric field exists, plot a 2D scatter
if len(numeric_candidates) > 1:
    plt.figure(figsize=(6,5))
    sns.scatterplot(data=df, x=numeric_candidates[0], y=numeric_candidates[1])
    plt.xlabel(numeric_candidates[0] + ' (@id)')
    plt.ylabel(numeric_candidates[1] + ' (@id)')
    plt.title(f"{numeric_candidates[0]} vs {numeric_candidates[1]}")
    plt.show()


## 6. Conclusion

- You loaded and explored the *FAIR^2* dataset using its Croissant schema, consistently referencing all entities by their `@id`.
- The notebook provided overviews of dataset record sets and their fields, data extraction based on `@id`, and common EDA procedures.
- Further analysis can leverage the `mlcroissant` library and the normalized, structured field access with Croissant for robust, reproducible science.